In [1]:
!pip -q install groq datasets sentence-transformers rank_bm25 openai huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 4.4 MB/s eta 0:00:00


In [ ]:
import os, json, re, time, random, unicodedata, itertools
import numpy as np, pandas as pd
from google.colab import userdata, drive

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# ---- OpenRouter (PRIMARY transport) ----
OPENROUTER_API_KEY = userdata.get("Open_R_Key")
assert OPENROUTER_API_KEY, "Add OPENROUTER_API_KEY to Colab Secrets"

# ---- Groq (fallback) ----
GROQ_KEYS = []
for i in range(1, 10):
    try:
        k = userdata.get(f"GROQ_API_KEY_{i}")
        if k: GROQ_KEYS.append(k)
    except Exception:
        pass

# ---- HF token (uppercase - required, see Legal session). Also the last-resort transport. ----
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"   # prevents download stalls
print(f"OpenRouter: ready | Groq fallback keys: {len(GROQ_KEYS)} | HF: "
      f"{'ready' if os.environ.get('HF_TOKEN') else 'MISSING'}")



drive.mount("/content/drive", force_remount=False)
SAVE_PATH = "/content/drive/MyDrive/RAGBench_Results/NEW_BIO_Aug"
os.makedirs(SAVE_PATH, exist_ok=True)
print("save path:", SAVE_PATH)

OpenRouter: ready | Groq fallback keys: 0 | HF: ready
Mounted at /content/drive
save path: /content/drive/MyDrive/RAGBench_Results/NEW_BIO_Aug


In [ ]:
import os

# Run just one experiment (optional) — set to a specific BIO-### id for a targeted
# rerun (same pattern as the CS-034..048 outage recovery). Accepts a comma-separated
# list and bare numbers: "BIO-031", "31", "BIO-031,BIO-032" all work.
# Leave blank to run/resume the full sweep and let checkpoint/resume skip what is done.
BIO_EXP_ONLY    = "BIO-007,BIO-015"      # e.g. "BIO-031"
BIO_FORCE_RERUN = True   # True to force a rerun even if already marked done

os.environ.pop("EXP_ONLY", None); os.environ.pop("FORCE_RERUN", None)
if BIO_EXP_ONLY:
    os.environ["EXP_ONLY"] = BIO_EXP_ONLY
if BIO_FORCE_RERUN:
    os.environ["FORCE_RERUN"] = "1"

# Export keys to environment for the standalone script (subprocess cannot read userdata)
os.environ["OPENROUTER_API_KEY"] = OPENROUTER_API_KEY
if GROQ_KEYS:
    os.environ["GROQ_API_KEY"]  = GROQ_KEYS[0]
    os.environ["GROQ_API_KEYS"] = ",".join(GROQ_KEYS)
os.environ["PROVIDER_ORDER"] = "openrouter,groq,hf"   # primary -> fallback -> last resort

In [6]:
import os, pathlib

# --- paths: LOCAL must NOT be on Drive, or you write to Drive twice ---
os.environ["SAVE_PATH"]  = "/content/bio_local"                              # scratch
os.environ["LOCAL_ROOT"] = "/content/bio_local"                              # fast local leg
os.environ["DRIVE_ROOT"] = "/content/drive/MyDrive/RAGBench_Results/NEW_BIO_Aug"          # Drive leg
pathlib.Path("/content/bio_local").mkdir(parents=True, exist_ok=True)

# --- secrets: subprocess can't read Colab userdata, so export them ---
#os.environ["GIT_PAT_V"]  = PUSH_TOKEN
os.environ["REPO_ROOT"]  = "/content/RAGBench-Capstone-Batch26"
os.environ["ENABLE_GIT"] = "1" if os.path.isdir("/content/RAGBench-Capstone-Batch26/.git") else "0"

# --- cadence + run controls ---
os.environ["CHECKPOINT_EVERY"]      = "5"    # local+Drive save every 5 examples
#os.environ["GIT_PUSH_EVERY_N_CKPT"] = "5"    # git push every 5 checkpoints (25 examples)
os.environ["N_EXAMPLES"]    = "200"          # 1 smoke -> 25 iterate -> 200 final
os.environ["GROQ_COOLDOWN"] = "1.0"
os.environ["GUARD_PAUSE"]   = "3"            # set to 3 for the N=200 run
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["JUDGE_ABORT_AFTER"] = "10"       # halt after 10 consecutive judge failures
os.environ["MIN_VALID_FRAC"]    = "0.80"     # judge coverage below this = not report-grade

DOMAIN = "biomedical"
os.environ["DOMAIN"] = DOMAIN
os.environ["DATASET_CONFIG"] = "covidqa"

# NOTE: biomedical's matrix keeps BOTH dense and hybrid retrieval (96 experiments),
# so DENSE_ONLY is intentionally left unset — don't force dense-only here.
os.environ.pop("DENSE_ONLY", None)
os.environ.pop("QUICK_SWEEP", None)

In [7]:
# ---- optional: what is still pending before you burn quota ----
import json, os
base = os.environ["LOCAL_ROOT"] + "/biomedical"
drv  = os.environ["DRIVE_ROOT"] + "/biomedical"
src  = base if os.path.exists(f"{base}/state_biomedical.json") else drv
try:
    state = json.load(open(f"{src}/state_biomedical.json"))
    reg   = json.load(open(f"{src}/experiments_biomedical.json"))
    ids   = sorted(set(reg.values()))
    done  = sorted(state.keys())
    print(f"done {len(done)}/{len(ids)}")
    print("pending:", ", ".join(i for i in ids if i not in state) or "none")
except FileNotFoundError:
    print("no state yet — this is a fresh sweep")

done 19/96
pending: BIO-020, BIO-021, BIO-022, BIO-023, BIO-024, BIO-025, BIO-026, BIO-027, BIO-028, BIO-029, BIO-030, BIO-031, BIO-032, BIO-033, BIO-034, BIO-035, BIO-036, BIO-037, BIO-038, BIO-039, BIO-040, BIO-041, BIO-042, BIO-043, BIO-044, BIO-045, BIO-046, BIO-047, BIO-048, BIO-049, BIO-050, BIO-051, BIO-052, BIO-053, BIO-054, BIO-055, BIO-056, BIO-057, BIO-058, BIO-059, BIO-060, BIO-061, BIO-062, BIO-063, BIO-064, BIO-065, BIO-066, BIO-067, BIO-068, BIO-069, BIO-070, BIO-071, BIO-072, BIO-073, BIO-074, BIO-075, BIO-076, BIO-077, BIO-078, BIO-079, BIO-080, BIO-081, BIO-082, BIO-083, BIO-084, BIO-085, BIO-086, BIO-087, BIO-088, BIO-089, BIO-090, BIO-091, BIO-092, BIO-093, BIO-094, BIO-095, BIO-096


In [ ]:
# ===== Resilient launcher: reconnect + resume, whatever went wrong. Handles all
# three cases with ONE cell -- a stalled subprocess, a crashed subprocess, or the
# whole Colab runtime having disconnected (drive unmounted, secrets/env vars gone).
# Re-run this cell for any of them.
#
# The one thing it can't recover on its own: if the runtime was fully replaced
# (not just reconnected -- a genuinely fresh VM), installed packages are gone too.
# You'll see that as an immediate "missing packages" stop below; re-run the
# `!pip install` cell once, then this one. =====
import os, sys, json, time, subprocess, threading, pathlib, importlib.util
from google.colab import userdata, drive

# ---- 1) Drive: idempotent, safe to call every time ----
drive.mount("/content/drive", force_remount=False)

# ---- 2) Secrets: only fetched if missing, so a same-session rerun (recovering
# from a stall) doesn't re-prompt -- but a fresh runtime gets them back either way ----
if not os.environ.get("OPENROUTER_API_KEY"):
    _or_key = None
    for _name in ("OPENROUTER_API_KEY", "OPEN_R_Key"):    # 2nd is a fallback alias --
        try:                                              # rename your Colab secret to
            _or_key = userdata.get(_name)                 # the first name when you can,
            if _or_key:                                   # so only one name is in play.
                break
        except Exception:
            pass
    assert _or_key, "Add OPENROUTER_API_KEY (or OPEN_R_Key) to Colab Secrets"
    os.environ["OPENROUTER_API_KEY"] = _or_key

if not os.environ.get("GROQ_API_KEYS"):
    _groq = []
    for i in range(1, 10):
        try:
            k = userdata.get(f"GROQ_API_KEY_{i}")
            if k: _groq.append(k)
        except Exception:
            pass
    if _groq:
        os.environ["GROQ_API_KEY"]  = _groq[0]
        os.environ["GROQ_API_KEYS"] = ",".join(_groq)

if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

if not os.environ.get("GIT_PAT_V"):
    try:
        os.environ["GIT_PAT_V"] = userdata.get("GIT_PAT_V")
    except Exception:
        pass   # git push is optional infra -- ENABLE_GIT below covers its absence

_ngroq = len(os.environ.get("GROQ_API_KEYS", "").split(",")) if os.environ.get("GROQ_API_KEYS") else 0
print(f"[resume] OpenRouter: {'ready' if os.environ.get('OPENROUTER_API_KEY') else 'MISSING'} | "
      f"Groq fallback keys: {_ngroq} | HF: {'ready' if os.environ.get('HF_TOKEN') else 'MISSING'}")

# ---- 3) Paths + run config: setdefault so anything you deliberately changed
# earlier this session (e.g. a temporary EXP_ONLY) is never silently clobbered ----
os.environ.setdefault("SAVE_PATH",  "/content/bio_local")
os.environ.setdefault("LOCAL_ROOT", "/content/bio_local")
os.environ.setdefault("DRIVE_ROOT", "/content/drive/MyDrive/RAGBench_Results/NEW_BIO_Aug")
os.environ.setdefault("REPO_ROOT",  "/content/RAGBench-Capstone-Batch26")
os.environ.setdefault("ENABLE_GIT", "1" if os.path.isdir(f"{os.environ['REPO_ROOT']}/.git") else "0")
os.environ.setdefault("CHECKPOINT_EVERY", "5")
os.environ.setdefault("GIT_PUSH_EVERY_N_CKPT", "5")
os.environ.setdefault("N_EXAMPLES", "200")      # final stage -- smoke(1) -> iterate(25) -> final(200)
os.environ.setdefault("GROQ_COOLDOWN", "1.0")
os.environ.setdefault("GUARD_PAUSE", "3")
os.environ.setdefault("HF_HOME", "/content/hf_cache")
os.environ.setdefault("JUDGE_ABORT_AFTER", "10")
os.environ.setdefault("MIN_VALID_FRAC", "0.80")
os.environ.setdefault("DOMAIN", "biomedical")
os.environ.setdefault("DATASET_CONFIG", "covidqa")
os.environ.setdefault("PROVIDER_ORDER", "openrouter,groq,hf")
pathlib.Path(os.environ["LOCAL_ROOT"]).mkdir(parents=True, exist_ok=True)
# targeted rerun (optional): set os.environ["EXP_ONLY"] = "BIO-031" / os.environ["FORCE_RERUN"] = "1"
# here, or in the run-control cell above, before running this cell.

SCRIPT        = "/content/biomedical_rag_standalone.py"
STALL_MIN     = 12      # no new stdout for this long -> treat as truly stuck (raise
                         # this if you see false-positive restarts during long backoffs)
POLL_SEC      = 5
MAX_RELAUNCH  = 25       # hard ceiling so a genuinely broken config can't loop forever
FAST_FAIL_SEC = 20       # an exit this fast usually means a real error, not a stall
FAST_FAIL_MAX = 3        # this many fast exits in a row -> stop instead of looping

if not os.path.exists(SCRIPT):
    dupes = sorted(pathlib.Path("/content").glob("biomedical_rag_standalone*.py"))
    hint = (f" Found instead: {[p.name for p in dupes]} -- these are almost always "
            f"repeat-download duplicates from your browser (the '(1)', '(2)'... suffix). "
            f"Delete all of them and upload one clean biomedical_rag_standalone.py."
            if dupes else " No matching file found at all -- upload it to /content/.")
    raise SystemExit(f"missing {SCRIPT}.{hint}")

_missing = [m for m in ("torch", "sentence_transformers", "datasets", "groq", "rank_bm25")
            if importlib.util.find_spec(m) is None]
if _missing:
    raise SystemExit(f"missing packages {_missing} -- this looks like a fresh VM (the runtime "
                      f"was replaced, not just reconnected). Re-run the '!pip install' cell "
                      f"once, then this one.")

# ---- heads-up only (does not block): flags an N_EXAMPLES change vs checkpointed state ----
base = os.environ["LOCAL_ROOT"] + "/biomedical"
drv  = os.environ["DRIVE_ROOT"] + "/biomedical"
src  = base if os.path.exists(f"{base}/state_biomedical.json") else drv
try:
    state  = json.load(open(f"{src}/state_biomedical.json"))
    seen_n = {rec.get("n_examples") for rec in state.values()}
    cur_n  = int(os.environ.get("N_EXAMPLES", "0"))
    if seen_n and cur_n not in seen_n:
        print(f"[resume] HEADS UP: N_EXAMPLES={cur_n} but checkpointed state was built at "
              f"n_examples={sorted(seen_n)}. This launch starts a FRESH set of 96 pending "
              f"experiments under a new fingerprint -- correct if you're deliberately moving "
              f"stages (e.g. iterate -> final); wrong if you meant to resume n_examples="
              f"{sorted(seen_n)[0]}.", flush=True)
    elif seen_n:
        print(f"[resume] {len(state)} experiment(s) already checkpointed at "
              f"n_examples={sorted(seen_n)} -- continuing", flush=True)
except FileNotFoundError:
    print("[resume] no state yet -- fresh sweep", flush=True)

def _launch():
    return subprocess.Popen(
        [sys.executable, "-u", SCRIPT, "biomedical"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, universal_newlines=True,
    )

def _pump(proc, box):
    for line in proc.stdout:
        print(line, end="", flush=True)
        box["last"] = time.time()
    box["eof"] = True

relaunches, fast_fails = 0, 0
while True:
    t0 = time.time()
    proc = _launch()
    box = {"last": time.time(), "eof": False}
    threading.Thread(target=_pump, args=(proc, box), daemon=True).start()

    try:
        while not (proc.poll() is not None and box["eof"]):
            if time.time() - box["last"] > STALL_MIN * 60:
                print(f"\n[watchdog] no output for {STALL_MIN} min -- killing and restarting", flush=True)
                proc.terminate()
                time.sleep(5)
                if proc.poll() is None:
                    proc.kill()
                break
            time.sleep(POLL_SEC)
    except KeyboardInterrupt:
        print("\n[watchdog] stopped by user -- terminating subprocess, not retrying", flush=True)
        proc.terminate()
        time.sleep(3)
        if proc.poll() is None:
            proc.kill()
        raise

    rc = proc.wait()
    ran = time.time() - t0

    if rc == 0:
        print(f"\n[watchdog] exited cleanly (rc=0) after {ran:.0f}s -- nothing left to do", flush=True)
        break

    relaunches += 1
    fast_fails = fast_fails + 1 if ran < FAST_FAIL_SEC else 0
    tag = "JudgeOutage (designed recovery path)" if rc == 2 else f"exit code {rc}"
    print(f"\n[watchdog] stopped -- {tag} after {ran:.0f}s (relaunch {relaunches}/{MAX_RELAUNCH})", flush=True)

    if fast_fails >= FAST_FAIL_MAX:
        print(f"[watchdog] {fast_fails} fast exits in a row (<{FAST_FAIL_SEC}s) -- looks like a real "
              f"config/credential problem, not a stall. Stopping instead of looping; check the log above.",
              flush=True)
        break
    if relaunches >= MAX_RELAUNCH:
        print(f"[watchdog] hit MAX_RELAUNCH={MAX_RELAUNCH} -- stopping. Re-run this cell once "
              f"you've checked what's failing.", flush=True)
        break

    time.sleep(10)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[resume] OpenRouter: ready | Groq fallback keys: 0 | HF: ready
[resume] HEADS UP: N_EXAMPLES=200 but checkpointed state was built at n_examples=[25]. This launch starts a FRESH set of 96 pending experiments under a new fingerprint -- correct if you're deliberately moving stages (e.g. iterate -> final); wrong if you meant to resume n_examples=25.
[select_device] importing torch (attempt 1/2)...
[drive] restored 28 file(s) from /content/drive/MyDrive/RAGBench_Results/NEW_BIO_Aug/biomedical
[cache] gen=353 judge=314
[data] loading HuggingFace rungalileo/ragbench/covidqa ...

Generating train split: 100%|██████████| 1252/1252 [00:00<00:00, 14197.41 examples/s]

Generating test split: 100%|██████████| 246/246 [00:00<00:00, 16155.43 examples/s]

Generating validation split: 100%|██████████| 267/267 [00:00<00:00, 17698.60 examples/s]
[data] HuggingFace rungalileo/ra

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ===== Biomedical report export — paste-and-run cell =====
import os
import pandas as pd

BASE           = "/content/drive/MyDrive/RAGBench_Results/NEW_BIO_Aug/biomedical"
N_FILTER       = None    # None = all sample sizes
MIN_VALID_FRAC = 0.80    # judge coverage below this = not report-grade
SORT_BY        = "adherence_auroc"
SHOW_EXP_ID    = True

COLUMNS = [
    ("Judge Model", "judge_model"), ("Embedding Model", "embedder"),
    ("Chunking Strategy", "chunk_config"), ("Generator LLM", "gen_model"),
    ("Retrieval Method", "retrieval"), ("Re-ranking Method", "rerank"),
    ("Context Ordering", "context_order"),
    ("TRACe Context Relevance \u2191", "context_relevance"),
    ("TRACe Context Utilization \u2191", "context_utilization"),
    ("TRACe Completeness \u2191", "completeness"),
    ("TRACe Adherence \u2191", "adherence"),
    ("Valid_Judge_response", "_valid"),
    ("Context Relevance RMSE \u2193", "context_relevance_rmse"),
    ("Context Utilization RMSE \u2193", "context_utilization_rmse"),
    ("Completeness RMSE \u2193", "completeness_rmse"),
    ("Adherence AUROC \u2191", "adherence_auroc"),
]

df = pd.read_csv(f"{BASE}/results_biomedical.csv")
print(f"master rows: {len(df)}")

if N_FILTER is not None:
    df = df[df["n_examples"].astype(str) == str(N_FILTER)]

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce", utc=True)
n0 = len(df)
df = df.sort_values("timestamp").drop_duplicates("exp_id", keep="last").reset_index(drop=True)
if n0 != len(df):
    print(f"de-duplicated {n0} -> {len(df)} (newest row per exp_id)")

df["judge_used"] = pd.to_numeric(df["judge_used"], errors="coerce").fillna(0.0)
df["n_examples"] = pd.to_numeric(df["n_examples"], errors="coerce").fillna(0).astype(int)
df["_valid"] = ((df["judge_used"] * df["n_examples"]).round().astype(int).astype(str)
                + "/" + df["n_examples"].astype(str))
df["Status"] = ["report-grade" if u >= MIN_VALID_FRAC else "LOW JUDGE COVERAGE"
                for u in df["judge_used"]]
df["_rank"]  = (df["Status"] != "report-grade").astype(int)
df["_s"]     = pd.to_numeric(df.get(SORT_BY), errors="coerce")
df = df.sort_values(["_rank", "_s"], ascending=[True, False], na_position="last")

out = pd.DataFrame()
if SHOW_EXP_ID:
    out["Experiment ID"] = df["exp_id"]
for label, src in COLUMNS:
    out[label] = df[src] if src in df.columns else ""
out["Status"] = df["Status"]

ok, bad = out[out.Status == "report-grade"], out[out.Status != "report-grade"]
for frame, name in ((out, "ALL"), (ok.drop(columns="Status"), ""),
                    (bad, "EXCLUDED")):
    suffix = f"_{name}" if name else ""
    frame.to_csv(f"{BASE}/report_biomedical{suffix}.csv",
                 index=False, encoding="utf-8-sig")   # BOM keeps up/down arrows in Excel

print(f"report-grade: {len(ok)} | low coverage: {len(bad)} | total: {len(out)}")
if len(bad):
    print("excluded: " + ", ".join(f"{e}({v})" for e, v in
                                   zip(bad["Experiment ID"], bad["Valid_Judge_response"])))

with pd.option_context("display.max_rows", None, "display.max_columns", None,
                       "display.width", 300):
    print("\n" + out.to_string(index=False))

# ---- current best config, derived fresh from this export -- don't hand-maintain a
# winner in prose, it goes stale exactly like this note used to (it claimed
# fixed_200w + cross_encoder, AUROC 0.6587; the real n=25 top row is
# sentence_4o1 + no rerank, AUROC 0.7727) ----
if len(ok):
    best_idx   = ok.index[0]
    best       = ok.loc[best_idx]
    scores     = df.loc[ok.index, "_s"]
    n_this     = int(df.loc[best_idx, "n_examples"])
    n_distinct = scores.nunique()
    n_tied     = int((scores == scores.iloc[0]).sum())
    print("\n" + "=" * 78)
    print(f"CURRENT BEST by {SORT_BY} -- {best['Experiment ID']}"
          + (f"  (tied with {n_tied - 1} other config(s))" if n_tied > 1 else ""))
    print(f"n_examples={n_this} | {len(ok)} report-grade rows | {n_distinct} distinct AUROC values")
    for label, _ in COLUMNS[:7]:
        print(f"  {label:22s} {best[label]}")
    print(f"  {'Adherence AUROC ↑':22s} {best['Adherence AUROC ↑']}")
    if n_this < 100:
        print(f"\nCAUTION: n={n_this} is small for AUROC (a rank statistic) -- only {n_distinct} "
              f"distinct values are reachable across these {len(ok)} rows, so ties are common and "
              f"this ranking is directional, not final. Re-check once n=200 lands.")
    print("=" * 78)
